# Neural Authorship Attribution: Hamilton vs Madison

This notebook trains a neural network classifier on the Federalist papers attributed to **HAMILTON** and **MADISON**, then predicts likely authors for **DISPUTED** and **COAUTHORED** papers.

In [44]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import SMOTE

from lexos.classification import fit_classifier
from lexos.dtm import DTM
from lexos.tokenizer import WhitespaceTokenizer

In [45]:
SEED = 42  # seed for reproducibility
np.random.seed(SEED)

base = Path.cwd()
search_roots = [base] + list(base.parents)

# locate papers directory
data_dir = next(
    (root / "fed_papers" for root in search_roots if (root / "fed_papers").exists()),
    None,
)

if data_dir is None:
    raise FileNotFoundError(
        "Could not locate 'fed_papers' from the current notebook location."
    )

print("Using data directory:", data_dir)

Using data directory: /home/mango/Lexos_Independant_Research/lexos/doc_src/docs/tutorials/classification/fed_papers


Separate files into two groups:
- Training set: HAMILTON and MADISON
- Unknown set: DISPUTED and COAUTHORED

In [46]:
train_dirs = ["HAMILTON", "MADISON"]
unknown_dirs = ["DISPUTED", "COAUTHORED"]

train_files = []
train_labels = []
for author in train_dirs:
    files = sorted((data_dir / author).glob("*.txt"))
    train_files.extend(files)
    train_labels.extend([author] * len(files))

unknown_files = []
unknown_sets = []
for subset in unknown_dirs:
    files = sorted((data_dir / subset).glob("*.txt"))
    unknown_files.extend(files)
    unknown_sets.extend([subset] * len(files))

print(f"Training docs: {len(train_files)}")
print(f"Unknown docs: {len(unknown_files)}")
print(pd.Series(train_labels).value_counts())

Training docs: 65
Unknown docs: 15
HAMILTON    51
MADISON     14
Name: count, dtype: int64


## Tokenization
- Lowercases text, filters to alphabetic tokens, and adds bigrams
- Tokenizes all training and unknown documents into token lists

In [47]:
from lexos.tokenizer import WhitespaceTokenizer
from lexos.tokenizer.ngrams import Ngrams

# Initialize tokenizers
ws_tokenizer = WhitespaceTokenizer()
ng = Ngrams(n=2)  # bigrams for better authorship attribution

train_token_lists = []
for path in train_files:
    text = path.read_text(encoding="utf-8", errors="ignore").lower()
    unigrams = [tok for tok in ws_tokenizer(text) if tok.isalpha()]
    bigrams = [
        f"{t1}_{t2}"
        for t1, t2 in ng.from_tokens(unigrams, output="tuples")
        if t1.isalpha() and t2.isalpha()
    ]
    train_token_lists.append(unigrams + bigrams)

unknown_token_lists = []
for path in unknown_files:
    text = path.read_text(encoding="utf-8", errors="ignore").lower()
    unigrams = [tok for tok in ws_tokenizer(text) if tok.isalpha()]
    bigrams = [
        f"{t1}_{t2}"
        for t1, t2 in ng.from_tokens(unigrams, output="tuples")
        if t1.isalpha() and t2.isalpha()
    ]
    unknown_token_lists.append(unigrams + bigrams)

if any(len(tokens) == 0 for tokens in train_token_lists):
    raise ValueError("At least one training document produced zero tokens.")

print("Example tokenized document length:", len(train_token_lists[0]))
print("Sample tokens:", train_token_lists[0][:20])

Example tokenized document length: 4449
Sample tokens: ['to', 'the', 'people', 'of', 'the', 'state', 'of', 'new', 'the', 'importance', 'of', 'the', 'in', 'a', 'commercial', 'is', 'one', 'of', 'those', 'points']


## Feature Extraction (Leakage-Safe Holdout Setup)
- Splits documents into train/test before fitting feature extraction
- Fits DTM vocabulary on holdout-train documents only (`min_df=2`)
- Transforms holdout-test docs with the fitted vocabulary
- Scales train/test with a scaler fitted on holdout-train only
- Uses SMOTE later on the scaled training split before fitting the neural net

In [48]:
all_token_lists = train_token_lists
all_doc_labels = [p.name for p in train_files]
y = np.array(train_labels)

indices = np.arange(len(all_token_lists))
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
    shuffle=True,
)

token_train = [all_token_lists[i] for i in train_idx]
token_test = [all_token_lists[i] for i in test_idx]
doc_labels_train = [all_doc_labels[i] for i in train_idx]
y_train = y[train_idx]
y_test = y[test_idx]

dtm_holdout = DTM()
X_train = dtm_holdout.fit_transform(token_train, labels=doc_labels_train, min_df=2)
X_test = dtm_holdout.transform(token_test)

scaler_holdout = StandardScaler(with_mean=False)
X_train_scaled = scaler_holdout.fit_transform(X_train)
X_test_scaled = scaler_holdout.transform(X_test)

print("Holdout train matrix shape:", X_train.shape)
print("Holdout test matrix shape:", X_test.shape)
print("Holdout vocabulary size:", len(dtm_holdout.vectorizer.terms_list))

Holdout train matrix shape: (52, 15161)
Holdout test matrix shape: (13, 15161)
Holdout vocabulary size: 15161


# Training & Holdout Evaluation

- Uses leakage-safe train/test features from the previous cell
- Applies SMOTE to the scaled training split before fitting
- Trains a smaller MLP neural network
- Evaluates with classification report, balanced accuracy, and macro F1
    - Accuracy: The percentage of total correct guesses.
    - Balanced Accuracy: The average of the accuracy for each class individually. This is your most important metric because it reveals if the model is ignoring Madison to get a "cheap" high score on Hamilton.
    - Macro F1: A score that balances precision and recall, treating Hamilton and Madison as equally important regardless of how many papers they each have. 
- Displays the confusion matrix for holdout performance

In [49]:
smote = SMOTE(random_state=SEED)
X_train_balanced, y_train_balanced = smote.fit_resample(
    X_train_scaled.toarray(), y_train
)

mlp_holdout = fit_classifier(
    feature_matrix=X_train_balanced,  # SMOTE-balanced training features
    target_labels=y_train_balanced,  # Corresponding balanced training labels
    model="mlp",  # Use multi-layer perceptron neural network
    hidden_layer_sizes=(64,),  # Single hidden layer with 64 neurons
    activation="relu",  # ReLU activation function for hidden layers
    solver="adam",  # Adam optimizer for weight updates
    alpha=1e-4,  # L2 regularization strength to prevent overfitting
    learning_rate_init=1e-3,  # Initial learning rate for the optimizer
    max_iter=1000,  # Maximum number of training iterations
    random_state=SEED,  # Fixed seed for reproducible results
)

y_pred = mlp_holdout.predict(X_test_scaled.toarray())
print(classification_report(y_test, y_pred, digits=4))
print(
    "Holdout balanced accuracy:",
    round(float(balanced_accuracy_score(y_test, y_pred)), 4),
)
print("Holdout macro F1:", round(float(f1_score(y_test, y_pred, average="macro")), 4))

cm = confusion_matrix(y_test, y_pred, labels=["HAMILTON", "MADISON"])
pd.DataFrame(
    cm,
    index=["true_HAMILTON", "true_MADISON"],
    columns=["pred_HAMILTON", "pred_MADISON"],
)

              precision    recall  f1-score   support

    HAMILTON     1.0000    0.9000    0.9474        10
     MADISON     0.7500    1.0000    0.8571         3

    accuracy                         0.9231        13
   macro avg     0.8750    0.9500    0.9023        13
weighted avg     0.9423    0.9231    0.9265        13

Holdout balanced accuracy: 0.95
Holdout macro F1: 0.9023


,pred_HAMILTON,pred_MADISON
true_HAMILTON,9,1
true_MADISON,0,3


# Cross-Validation (Leakage-Safe)

- Runs 5-fold stratified CV on all labeled known-author documents
    - each fold contains the same 80/20 balance of Hamilton and Madison papers as the original dataset
- Re-fits DTM + scaler inside each fold (no preprocessing leakage)
- Applies SMOTE inside each fold before training
- Trains a fresh, smaller MLP each fold and reports accuracy, balanced accuracy, and macro F1

In [50]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_acc = []
cv_bal_acc = []
cv_f1_macro = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(all_token_lists, y), start=1):
    fold_train_tokens = [all_token_lists[i] for i in tr_idx]
    fold_valid_tokens = [all_token_lists[i] for i in va_idx]
    fold_train_doc_labels = [all_doc_labels[i] for i in tr_idx]
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    dtm_fold = DTM()
    X_tr = dtm_fold.fit_transform(
        fold_train_tokens, labels=fold_train_doc_labels, min_df=2
    )
    X_va = dtm_fold.transform(fold_valid_tokens)

    scaler_fold = StandardScaler(with_mean=False)
    X_tr_scaled = scaler_fold.fit_transform(X_tr)
    X_va_scaled = scaler_fold.transform(X_va)

    smote = SMOTE(random_state=SEED)
    X_tr_balanced, y_tr_balanced = smote.fit_resample(X_tr_scaled.toarray(), y_tr)

    fold_model = fit_classifier(
        feature_matrix=X_tr_balanced,
        target_labels=y_tr_balanced,
        model="mlp",
        hidden_layer_sizes=(64,),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=1000,
        random_state=SEED,
    )

    fold_pred = fold_model.predict(X_va_scaled.toarray())
    cv_acc.append(accuracy_score(y_va, fold_pred))
    cv_bal_acc.append(balanced_accuracy_score(y_va, fold_pred))
    cv_f1_macro.append(f1_score(y_va, fold_pred, average="macro"))

print("5-fold CV accuracy:", np.round(cv_acc, 4))
print("Mean CV accuracy:", round(float(np.mean(cv_acc)), 4))
print("5-fold CV balanced accuracy:", np.round(cv_bal_acc, 4))
print("Mean CV balanced accuracy:", round(float(np.mean(cv_bal_acc)), 4))
print("5-fold CV macro F1:", np.round(cv_f1_macro, 4))
print("Mean CV macro F1:", round(float(np.mean(cv_f1_macro)), 4))

5-fold CV accuracy: [0.9231 0.7692 0.8462 0.7692 1.    ]
Mean CV accuracy: 0.8615
5-fold CV balanced accuracy: [0.75   0.85   0.6667 0.5    1.    ]
Mean CV balanced accuracy: 0.7533
5-fold CV macro F1: [0.8116 0.7451 0.7045 0.4348 1.    ]
Mean CV macro F1: 0.7392


# Inference on Unknown Papers (Final Model Refit)

- Re-fits DTM and scaler on all known labeled documents
- Applies SMOTE before training the final MLP
- Transforms disputed/coauthored papers with the final vocabulary and scaler
- Predicts authors and returns confidence scores

In [51]:
dtm_final = DTM()
X_full = dtm_final.fit_transform(all_token_lists, labels=all_doc_labels, min_df=2)

scaler_final = StandardScaler(with_mean=False)
X_full_scaled = scaler_final.fit_transform(X_full)

smote = SMOTE(random_state=SEED)
X_full_balanced, y_balanced = smote.fit_resample(X_full_scaled.toarray(), y)

mlp_final = fit_classifier(
    feature_matrix=X_full_balanced,
    target_labels=y_balanced,
    model="mlp",
    hidden_layer_sizes=(64,),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    learning_rate_init=1e-3,
    max_iter=1000,
    random_state=SEED,
)

X_unknown = dtm_final.transform(unknown_token_lists)
X_unknown_scaled = scaler_final.transform(X_unknown)
unknown_pred = mlp_final.predict(X_unknown_scaled.toarray())

results = (
    pd.DataFrame(
        {
            "document": [p.name for p in unknown_files],
            "set": unknown_sets,
            "predicted_author": unknown_pred,
        }
    )
    .sort_values(["set", "document"])
    .reset_index(drop=True)
)

if hasattr(mlp_final, "predict_proba"):
    proba = mlp_final.predict_proba(X_unknown_scaled.toarray())
    class_to_col = {label: idx for idx, label in enumerate(mlp_final.classes_)}
    results["p_hamilton"] = proba[:, class_to_col["HAMILTON"]]
    results["p_madison"] = proba[:, class_to_col["MADISON"]]

results

,document,set,predicted_author,p_hamilton,p_madison
0,FED_18_C.txt,COAUTHORED,MADISON,0.022633,0.977367
1,FED_19_C.txt,COAUTHORED,MADISON,0.745548,0.254452
2,FED_20_C.txt,COAUTHORED,MADISON,0.000194,0.999806
3,FED_49_D.txt,DISPUTED,MADISON,0.019389,0.980611
4,FED_50_D.txt,DISPUTED,HAMILTON,0.021899,0.978101
5,FED_51_D.txt,DISPUTED,MADISON,0.002919,0.997081
6,FED_52_D.txt,DISPUTED,MADISON,0.009568,0.990432
7,FED_53_D.txt,DISPUTED,MADISON,0.001228,0.998772
8,FED_54_D.txt,DISPUTED,MADISON,0.031775,0.968225
9,FED_55_D.txt,DISPUTED,MADISON,0.030443,0.969557


# Save Results
- Exports predictions to CSV for downstream analysis

In [52]:
output_csv = data_dir / "neural_authorship_predictions.csv"
results.to_csv(output_csv, index=False)
print("Saved predictions:", output_csv)

Saved predictions: /home/mango/Lexos_Independant_Research/lexos/doc_src/docs/tutorials/classification/fed_papers/neural_authorship_predictions.csv
